In [1]:
%pip install -q pandas numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


# 04. Eras and history: orders as a lens on history

## How to read this notebook
An executive order count is not a measure of how important a President was. It is a measure of how often that President chose this tool, and what kind of work that tool was doing at the time. A spike can mean a crisis, a new program, or just a change in paperwork habits. This notebook ties spikes to dated events, and says plainly when the link is only timing, not proof of cause.

## Events used, with dates
New Deal: Franklin D. Roosevelt takes office March 4, 1933. Great Depression already underway since 1929.
World War II: United States enters the war December 1941, after Pearl Harbor, December 7, 1941.
Civil rights era: Brown v. Board of Education 1954, Civil Rights Act 1964, Voting Rights Act 1965.
Cold War: roughly 1947 to 1991, from the Truman Doctrine in 1947 to the dissolution of the Soviet Union in December 1991.
Post-9/11: September 11, 2001 attacks, followed by homeland security orders in late 2001.
Pandemic: COVID-19 declared a pandemic March 11, 2020, by the World Health Organization.
Recent terms: data run through October 22, 2026, so Trump second term figures are partial, not a full term total.

In [2]:
import pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120
from pathlib import Path
BASE = Path("..").resolve()
CH = BASE / "charts"
CH.mkdir(exist_ok=True)
DF = pd.read_csv(BASE / "data" / "processed" / "executive_orders.csv", parse_dates=["date_signed"])
print("Rows:", len(DF), "| Years:", DF["year"].min(), "to", DF["year"].max())

Rows: 10879 | Years: 1791 to 2026


In [3]:
def window(a, b, label):
    sub = DF[(DF["year"] >= a) & (DF["year"] <= b)]
    print(f"{label} ({a} to {b}): {len(sub)} orders, {len(sub) / (b - a + 1):.1f} per year")
    print(" Top categories:", sub.groupby("category").size().sort_values(ascending=False).head(4).to_dict())
    print()
window(1933, 1938, "New Deal first years")
window(1941, 1945, "World War II, United States at war")
window(1954, 1968, "Civil rights era")
window(1947, 1991, "Cold War, broad window")
window(2001, 2003, "Post-9/11 first years")
window(2020, 2021, "Pandemic years")
window(2025, 2026, "Trump second term, partial, through Oct 22 2026")
fig, ax = plt.subplots(figsize=(12, 5))
by_year = DF.groupby("year").size()
by_year.plot(ax=ax, color="#5b7fa6")
for x, lab in [(1933, "New Deal"), (1941, "WWII"), (1964, "Civil Rights Act"), (2001, "9/11"), (2020, "Pandemic"), (2025, "Trump 2nd term")]:
    ax.axvline(x, color="#e08a3c", linestyle="--", linewidth=1)
    ax.text(x, by_year.max() * 0.92, lab, rotation=90, va="top", fontsize=8)
ax.set_title("Orders per year with historical markers")
ax.set_ylabel("Orders")
plt.tight_layout()
plt.savefig(CH / "04_orders_with_markers.png")
plt.show()

New Deal first years (1933 to 1938): 2294 orders, 382.3 per year
 Top categories: {'Other or General': 896, 'Land and Public Lands': 617, 'Government Administration': 202, 'Emergency and Disaster': 108}

World War II, United States at war (1941 to 1945): 1062 orders, 212.4 per year
 Top categories: {'Defense and Military': 344, 'Other or General': 254, 'Government Administration': 92, 'Land and Public Lands': 92}

Civil rights era (1954 to 1968): 934 orders, 62.3 per year
 Top categories: {'Other or General': 172, 'Economy and Finance': 162, 'Government Administration': 156, 'Defense and Military': 106}

Cold War, broad window (1947 to 1991): 2979 orders, 66.2 per year
 Top categories: {'Other or General': 742, 'Government Administration': 480, 'Economy and Finance': 387, 'Defense and Military': 331}

Post-9/11 first years (2001 to 2003): 138 orders, 46.0 per year
 Top categories: {'Other or General': 41, 'Government Administration': 25, 'Economy and Finance': 12, 'Land and Public Land

In [4]:
print("New Deal and World War II, year by year:")
print(DF[(DF["year"] >= 1933) & (DF["year"] <= 1945)].groupby("year").size().to_string())
print()
print("What changed in topic mix? Category shares, New Deal vs World War II:")
for label, a, b in [("New Deal 1933-1938", 1933, 1938), ("World War II 1941-1945", 1941, 1945)]:
    sub = DF[(DF["year"] >= a) & (DF["year"] <= b)]
    share = (sub.groupby("category").size() / len(sub) * 100).sort_values(ascending=False).head(6)
    print(label, share.round(1).to_dict())
fig, ax = plt.subplots(figsize=(10, 5))
sub = DF[(DF["year"] >= 1933) & (DF["year"] <= 1945)]
sub.groupby("year").size().plot(kind="bar", ax=ax, color="#5b7fa6")
ax.set_title("Orders per year, 1933 to 1945")
ax.set_ylabel("Orders")
plt.tight_layout()
plt.savefig(CH / "04_new_deal_wwii.png")
plt.show()

New Deal and World War II, year by year:
year
1933    657
1934    472
1935    384
1936    275
1937    257
1938    249
1939    287
1940    309
1941    382
1942    290
1943    122
1944    100
1945    168

What changed in topic mix? Category shares, New Deal vs World War II:
New Deal 1933-1938 {'Other or General': 39.1, 'Land and Public Lands': 26.9, 'Government Administration': 8.8, 'Emergency and Disaster': 4.7, 'Economy and Finance': 4.5, 'Defense and Military': 4.1}
World War II 1941-1945 {'Defense and Military': 32.4, 'Other or General': 23.9, 'Government Administration': 8.7, 'Land and Public Lands': 8.7, 'Economy and Finance': 5.8, 'Foreign Policy and Trade': 4.8}


In [5]:
print("Civil rights era, orders whose titles mention equality, discrimination, or civil rights:")
mask = DF["title_clean"].str.lower().str.contains("equal|discriminat|civil rights", na=False)
cr = DF[mask & (DF["year"] >= 1954) & (DF["year"] <= 1968)]
print("Count in window:", len(cr))
print(cr[["executive_order_number", "date_signed", "president", "title_clean"]].head(20).to_string(index=False))
print()
print("Examples found in the dataset by EO number search:")
for eo in ["8802", "9981", "10925", "11246", "11478"]:
    sub = DF[DF["executive_order_number"].astype(str) == eo + ".0"]
    if len(sub):
        r = sub.iloc[0]
        print(f"EO {eo}, {r['date_signed'].date()}, {r['president']}: {r['title_clean']}")
    else:
        print(f"EO {eo}: not found as a row in this dataset")

Civil rights era, orders whose titles mention equality, discrimination, or civil rights:
Count in window: 18
 executive_order_number date_signed            president                                                                                                                                               title_clean
                10557.0  1954-09-03 Dwight D. Eisenhower                                                       Approving the Revised Provision in Government Contracts Relating to Nondiscrimination in Employment
                10925.0  1961-03-06      John F. Kennedy                                                                                    Establishing the President's Committee on Equal Employment Opportunity
                11063.0  1962-11-20      John F. Kennedy                                                                                                                              Equal Opportunity in Housing
                11114.0  1963-06-22      John F

In [6]:
print("Post-9/11: orders in late 2001 and 2002, top categories:")
sub = DF[(DF["year"] >= 2001) & (DF["year"] <= 2003)]
print(sub.groupby("category").size().sort_values(ascending=False).head(8).to_string())
print()
print("Homeland security in titles, 2001 to 2003:")
hs = sub[sub["title_clean"].str.lower().str.contains("homeland", na=False)]
print(hs[["executive_order_number", "date_signed", "president", "title_clean"]].to_string(index=False))
print()
print("Pandemic years, COVID in titles:")
cov = DF[DF["title_clean"].str.lower().str.contains("covid|coronavirus", na=False)]
print(cov.groupby("year").size().to_string())
print(cov[["executive_order_number", "date_signed", "president", "title_clean"]].head(15).to_string(index=False))

Post-9/11: orders in late 2001 and 2002, top categories:
category
Other or General             41
Government Administration    25
Economy and Finance          12
Land and Public Lands        11
Education and Science         9
Emergency and Disaster        7
Environment and Energy        6
Defense and Military          6

Homeland security in titles, 2001 to 2003:
 executive_order_number date_signed      president                                                                                                                                  title_clean
                13228.0  2001-10-08 George W. Bush                                                               Establishing the Office of Homeland Security and the Homeland Security Council
                13260.0  2002-03-19 George W. Bush                                  Establishing the President's Homeland Security Advisory Council and Senior Advisory Committees for Homeland
                13267.0  2002-06-20 George W. Bush        

In [7]:
print("Recent terms, orders per year, 2017 to 2026:")
print(DF[DF["year"] >= 2017].groupby(["year", "president"]).size().to_string())
fig, ax = plt.subplots(figsize=(10, 5))
DF[DF["year"] >= 2017].groupby("year").size().plot(kind="bar", ax=ax, color="#5b7fa6")
ax.set_title("Orders per year, 2017 to 2026 (2026 is partial, through Oct 22)")
ax.set_ylabel("Orders")
plt.tight_layout()
plt.savefig(CH / "04_recent_years.png")
plt.show()
print()
print("Caution: 2025 is a full calendar year in the data except for late December, while 2026 stops at October 22. Do not compare 2026 directly with a full year.")

Recent terms, orders per year, 2017 to 2026:
year  president      
2017  Barack Obama         7
      Donald J. Trump     55
2018  Donald J. Trump     37
2019  Donald J. Trump     45
2020  Donald J. Trump     69
2021  Donald J. Trump     14
      Joseph R. Biden     77
2022  Joseph R. Biden     29
2023  Joseph R. Biden     24
2024  Joseph R. Biden     19
2025  Donald J. Trump    225
      Joseph R. Biden     13
2026  Donald J. Trump     67



Caution: 2025 is a full calendar year in the data except for late December, while 2026 stops at October 22. Do not compare 2026 directly with a full year.


## What the lens shows, and what it does not
It shows that order volume follows crises and new administrations, especially first years. It does not show that orders caused those events, or that a high count means a strong President. Roosevelt's huge count also reflects thousands of routine orders that modern Presidents handle through other channels. The honest story is: the tool stayed the same in name, but its everyday use changed a lot.